In [4]:
import pandas as pd
import numpy as np

# Load Data

In [6]:
df = pd.read_csv("/content/Task 3 and 4_Loan_Data.csv")
df.head()

,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
0,8153374,0,5221.545193,3915.471226,78039.38546,5,605,0
1,7442532,5,1958.928726,8228.752520,26648.43525,2,572,1
2,2256073,0,3363.009259,2027.830850,65866.71246,4,602,0
3,4885975,0,4766.648001,2501.730397,74356.88347,5,612,0
4,4700614,1,1345.827718,1768.826187,23448.32631,6,631,0


# Creating Binns
Using MSE & DP

In [8]:
scores = np.sort(df['fico_score'].unique())
N = len(scores)
print(scores, N)

[408 409 418 425 438 440 441 444 447 449 450 455 456 459 460 461 462 463
 464 465 466 467 468 469 470 471 472 473 474 475 476 477 479 480 481 482
 483 484 485 486 487 488 489 490 491 492 493 494 495 496 497 498 499 500
 501 502 503 504 505 506 507 508 509 510 511 512 513 514 515 516 517 518
 519 520 521 522 523 524 525 526 527 528 529 530 531 532 533 534 535 536
 537 538 539 540 541 542 543 544 545 546 547 548 549 550 551 552 553 554
 555 556 557 558 559 560 561 562 563 564 565 566 567 568 569 570 571 572
 573 574 575 576 577 578 579 580 581 582 583 584 585 586 587 588 589 590
 591 592 593 594 595 596 597 598 599 600 601 602 603 604 605 606 607 608
 609 610 611 612 613 614 615 616 617 618 619 620 621 622 623 624 625 626
 627 628 629 630 631 632 633 634 635 636 637 638 639 640 641 642 643 644
 645 646 647 648 649 650 651 652 653 654 655 656 657 658 659 660 661 662
 663 664 665 666 667 668 669 670 671 672 673 674 675 676 677 678 679 680
 681 682 683 684 685 686 687 688 689 690 691 692 69

In [9]:
prefix_sum  = np.concatenate(([0], np.cumsum(scores)))
prefix_sum2 = np.concatenate(([0], np.cumsum(scores ** 2)))

In [10]:
def mse(i, j):
    """MSE of scores[i..j] in O(1) using prefix sums"""
    n    = j - i + 1
    s    = prefix_sum[j+1]  - prefix_sum[i]
    s2   = prefix_sum2[j+1] - prefix_sum2[i]
    mean = s / n
    return s2 - n * mean ** 2

In [11]:
NUM_BUCKETS = 10

dp = np.full((NUM_BUCKETS + 1, N), np.inf)
bt = np.zeros((NUM_BUCKETS + 1, N), dtype=int)

In [13]:
# Base case: bucket 1 covering scores[0..i]
for i in range(N):
    dp[1][i] = mse(0, i)
    bt[1][i] = 0

In [14]:
for b in range(2, NUM_BUCKETS + 1):
    for i in range(b - 1, N):

        # Try EVERY possible last split point m
        for m in range(b - 1, i + 1):
            left  = dp[b-1][m-1]     #Reuse
            right = mse(m, i)
            total = left + right

            if total < dp[b][i]:
                dp[b][i] = total
                bt[b][i] = m

In [15]:
def get_boundaries(bt, scores, num_buckets):
    N = len(scores)
    starts = []
    i = N - 1
    b = num_buckets
    while b > 0:
        starts.append(bt[b][i])
        i = bt[b][i] - 1
        b -= 1
    starts.reverse()
    return [scores[s] for s in starts]

In [16]:
boundaries = get_boundaries(bt, scores, NUM_BUCKETS)
print(boundaries)

[np.int64(408), np.int64(472), np.int64(512), np.int64(552), np.int64(592), np.int64(632), np.int64(672), np.int64(712), np.int64(752), np.int64(792)]


In [17]:
def fico_to_rating(fico_score, boundaries):
    """
    Lower FICO = higher rating number = worse credit
    Higher FICO = lower rating number = better credit
    """
    for i, b in enumerate(boundaries[1:], 1):
        if fico_score < b:
            return i
    return len(boundaries)

In [19]:
df["FICO_Rating"] = df["fico_score"].apply(lambda x: fico_to_rating(x, boundaries))

# Result: FICO rating

In [20]:
df

,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default,FICO_Rating
0,8153374,0,5221.545193,3915.471226,78039.38546,5,605,0,5
1,7442532,5,1958.928726,8228.752520,26648.43525,2,572,1,4
2,2256073,0,3363.009259,2027.830850,65866.71246,4,602,0,5
3,4885975,0,4766.648001,2501.730397,74356.88347,5,612,0,5
4,4700614,1,1345.827718,1768.826187,23448.32631,6,631,0,5
...,...,...,...,...,...,...,...,...,...
9995,3972488,0,3033.647103,2553.733144,42691.62787,5,697,0,7
9996,6184073,1,4146.239304,5458.163525,79969.50521,8,615,0,5
9997,6694516,2,3088.223727,4813.090925,38192.67591,5,596,0,5
9998,3942961,0,3288.901666,1043.099660,50929.37206,2,647,0,6
